# Intro

This notebook is for running the LLM as a judge scoring for the OSQs. 

Framework:
- Pull in the OSQ response data
- (Optional) list of judge models -- default only CXhatgpt5 or Chatgpt4o-mini
- List of Prompts
  - (Currently Omitted) Binary Correct/Incorrect
  - Scoring
    - Rubric 1 (from the MCQ -> OSQ conversion)
    - (Omitted) Rubric 2 (outside of recommended from conversion, e.g. from other sources)

# Configuration

In [32]:
# ================================================
# Phase 5 — Append-Only Judging (Resumable, Single JSONL, Progress)
# ================================================
from pathlib import Path
from datetime import datetime
from statistics import mean
from tqdm.auto import tqdm
from openai import OpenAI
import os, json

# -----------------------------
# CONFIG
# -----------------------------
TASK_NAME    = "sysengbench-osq"
JUDGE_MODEL  = "openai/gpt-5"
TEMPERATURE  = 0.0
MAX_TOKENS   = 2000
SAMPLE_N     = 3          # 0 = judge ALL samples; else judge first N (for quick tests)

# Paths (this notebook under: src/phase5_llm_as_a_judge/)
PHASE4_ROOT  = Path("../phase4_inference/output") / TASK_NAME
# PHASE5_ROOT  = Path(".") / TASK_NAME  # mirror structure in phase5
PHASE5_ROOT  = Path(".") / f"{TASK_NAME}-llm-judge" # Append "-llm-judge" to keep Phase 5 artifacts separate and consistent
PHASE5_ROOT.mkdir(parents=True, exist_ok=True)

# OpenRouter client
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)
print("OPENROUTER_API_KEY present:", bool(os.getenv("OPENROUTER_API_KEY")))

OPENROUTER_API_KEY present: True


## Optional Troubleshooting Directory

In [ ]:
# OPTIONAL TROUBLEHSOOTING CELL
from pathlib import Path
import json

p = Path("../phase4_inference/output/sysengbench-osq/gemma3__27b")  # or whichever model dir
f = sorted(p.glob("samples_*.jsonl"))[-1]  # newest
print(f"File = {f}")

with open(f,"r",encoding="utf-8") as fh:
    for i,line in enumerate(fh):
        if i>2: break
        print(json.loads(line))


File = ..\phase4_inference\downloaded_output\sysengbench-osq\gemma3__27b\samples_sysengbench-osq_2025-10-01T04-26-37.597737.jsonl
{'doc_id': 0, 'doc': {'Question ID': 1, 'Tags': 'Introduction to risk', 'INCOSE Handbook Category': 'INCOSEHandbook/Systems Engineering Overview/System Concepts and Structures', 'question': 'What best describes the concept of uncertainty in systems engineering?', 'choiceA': 'The process of systematically improving and optimizing a system for efficiency.', 'choiceB': 'The condition where the outcomes of system functions are not predictable due to lack of information or variability.', 'choiceC': 'A method for analyzing the costs and benefits of a system over its lifecycle.', 'choiceD': 'The act of integrating different system components into a cohesive whole.', 'answer': 'B', 'label': 1, 'Justification': 'Uncertainty in systems engineering refers to the unpredictability of outcomes due to insufficient information or inherent variability within the system or it

# Check Number of Judgings to Perform

## Simple Table

In [22]:
# Would be nice to have a cell that checks for how many samples have to be judged and have already been judged, to avoid re-judging them.

import json
import pandas as pd
from pathlib import Path

def build_judge_progress_matrix(
    phase4_root: str,
    phase5_root: str,
    task_name: str
) -> pd.DataFrame:
    """
    Build a matrix that summarizes judge progress for each model.

    Columns:
        model_name       – filesystem-safe model folder
        ollama_name      – restored with ':' instead of '__'
        judge_status     – not started | partial | complete
        judged_count     – # of judged OSQ responses so far
        total_samples    – total OSQ responses from Phase 4 output
        progress_fraction – judged_count / total_samples
    """

    p4 = Path(phase4_root)
    p5 = Path(phase5_root)

    if not p4.exists():
        raise FileNotFoundError(f"Phase 4 directory missing: {phase4_root}")

    model_folders = sorted([d.name for d in (p4 / task_name).iterdir() if d.is_dir()])

    rows = []

    for model in model_folders:
        model_p4_dir = p4 / task_name / model
        model_p5_dir = p5 / model

        # --------------------------------
        # 1. Count TOTAL samples (Phase 4)
        # --------------------------------

        # Find a samples_<task>_<ts>.jsonl file
        sample_files = [
            f for f in model_p4_dir.iterdir()
            if f.name.startswith(f"samples_{task_name}_") and f.suffix == ".jsonl"
        ]

        total_samples = 0
        if sample_files:
            # Usually only 1 samples file, but loop safely
            for sf in sample_files:
                with open(sf, "r", encoding="utf-8") as fh:
                    total_samples = sum(1 for _ in fh)
                break  # take first

        # --------------------------------
        # 2. Count JUDGED samples (Phase 5)
        # --------------------------------

        judged_count = 0

        if model_p5_dir.exists():
            judge_files = [
                f for f in model_p5_dir.iterdir()
                if f.name.startswith("samples_") and "__" in f.name and f.suffix == ".jsonl"
            ]

            for jf in judge_files:
                with open(jf, "r", encoding="utf-8") as fh:
                    judged_count += sum(1 for _ in fh)

        # --------------------------------
        # 3. Compute judge status
        # --------------------------------

        if total_samples == 0:
            judge_status = "not started"
        else:
            if judged_count == 0:
                judge_status = "not started"
            elif judged_count < total_samples:
                judge_status = "partial"
            else:
                judge_status = "complete"

        rows.append({
            "model_name": model,
            "ollama_name": model.replace("__", ":"),
            "judge_status": judge_status,
            "judged_count": judged_count,
            "total_samples": total_samples,
            "progress_fraction": (
                judged_count / total_samples if total_samples > 0 else 0.0
            ),
        })

    df = pd.DataFrame(rows)
    return df.sort_values("model_name")


In [23]:
df = build_judge_progress_matrix(
    phase4_root=str(PHASE4_ROOT.parent),
    phase5_root=str(PHASE5_ROOT),
    task_name=TASK_NAME
)

display(df)

,model_name,ollama_name,judge_status,judged_count,total_samples,progress_fraction
0,gemma3__27b,gemma3:27b,partial,3,845,0.00355
1,gemma3__4b,gemma3:4b,partial,3,845,0.00355


## (Not working, add later) Adding a column for the judge model name

In [18]:
import json
import pandas as pd
from pathlib import Path


# ------------------------------------------------------
# Helper: Extract judge metadata from a Phase-5 JSONL file
# ------------------------------------------------------

def parse_judge_metadata_from_file(path: Path):
    """
    Reads the FIRST JSONL entry in a Phase-5 judge file
    and extracts judge metadata (judge model, temperature, timestamp).

    Returns dict:
        {
            "judge_model": str | None,
            "temperature": float | None,
            "timestamp": str | None,
        }
    """
    try:
        with open(path, "r", encoding="utf-8") as fh:
            first_line = fh.readline().strip()
            if not first_line:
                return {"judge_model": None, "temperature": None, "timestamp": None}

            data = json.loads(first_line)
            meta = data.get("judge_metadata", {})

            return {
                "judge_model": meta.get("judge_model"),
                "temperature": meta.get("temperature"),
                "timestamp": meta.get("timestamp"),
            }

    except Exception as e:
        print(f"WARNING: Could not parse metadata from {path}: {e}")
        return {"judge_model": None, "temperature": None, "timestamp": None}



# ------------------------------------------------------
# Main function: Build the multi-judge progress table
# ------------------------------------------------------

def build_remaining_judging_table(
    phase4_root: str,
    phase5_root: str,
    task_name: str
) -> pd.DataFrame:
    """
    Creates a multi-judge progress table.

    Output columns:
        model_name
        ollama_name
        judge_name
        judge_temperature
        judge_timestamp
        total_samples
        judged_count
        remaining_to_judge
        judge_status
        need_to_judge
        judge_file
    """

    p4 = Path(phase4_root)
    p5 = Path(phase5_root)

    # Models from Phase-4
    model_folders = sorted([d.name for d in (p4 / task_name).iterdir() if d.is_dir()])

    rows = []

    for model in model_folders:
        model_p4_dir = p4 / task_name / model
        model_p5_dir = p5 / model

        # ------------------------
        # 1. Count total samples
        # ------------------------

        sample_files = [
            f for f in model_p4_dir.iterdir()
            if f.name.startswith(f"samples_{task_name}_") and f.suffix == ".jsonl"
        ]

        total_samples = 0
        if sample_files:
            with open(sample_files[0], "r", encoding="utf-8") as fh:
                total_samples = sum(1 for _ in fh)

        # ------------------------
        # 2. Read judge files
        # ------------------------

        judge_entries = []

        if model_p5_dir.exists():
            judge_files = [
                f for f in model_p5_dir.iterdir()
                if f.name.startswith("samples__") and f.suffix == ".jsonl"
            ]

            for jf in judge_files:
                # Count judged samples
                with open(jf, "r", encoding="utf-8") as fh:
                    judged_count = sum(1 for _ in fh)

                # Extract judge metadata
                meta = parse_judge_metadata_from_file(jf)

                judge_entries.append({
                    "judge_name": meta["judge_model"] or "unknown",
                    "judge_temperature": meta["temperature"],
                    "judge_timestamp": meta["timestamp"],
                    "judged_count": judged_count,
                    "judge_file": jf
                })

        # If no judge files yet, create a placeholder row
        if not judge_entries:
            judge_entries = [{
                "judge_name": None,
                "judge_temperature": None,
                "judge_timestamp": None,
                "judged_count": 0,
                "judge_file": None
            }]

        # ------------------------
        # 3. Build rows (one per judge)
        # ------------------------

        for entry in judge_entries:
            judged = entry["judged_count"]
            remaining = max(0, total_samples - judged)

            # Status logic
            if total_samples == 0:
                status = "no samples"
            elif judged == 0:
                status = "not started"
            elif judged < total_samples:
                status = "partial"
            else:
                status = "complete"

            rows.append({
                "model_name": model,
                "ollama_name": model.replace("__", ":"),

                "judge_name": entry["judge_name"],
                "judge_temperature": entry["judge_temperature"],
                "judge_timestamp": entry["judge_timestamp"],

                "total_samples": total_samples,
                "judged_count": judged,
                "remaining_to_judge": remaining,
                "judge_status": status,
                "need_to_judge": remaining > 0,

                "judge_file": entry["judge_file"],
            })

    df = pd.DataFrame(rows)
    return df.sort_values(["model_name", "judge_name"], na_position="last")


In [20]:
rem = build_remaining_judging_table(
    phase4_root=PHASE4_ROOT,
    phase5_root=PHASE5_ROOT,
    task_name=TASK_NAME
)

display(rem)


FileNotFoundError: [WinError 3] The system cannot find the path specified: '..\\phase4_inference\\downloaded_output\\sysengbench-osq\\sysengbench-osq'

# LLM-as-a-Judge Inferencing

## Judge Prompt

In [33]:
# -----------------------------
# Judge prompt (0–100 via five 0–20 dims) — braces escaped for .format()
# -----------------------------
JUDGE_PROMPT = """You are an expert systems engineering educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of systems engineering concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{{
  "technical_accuracy": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "conceptual_understanding": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "completeness": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "clarity_organization": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "professional_relevance": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "overall_score": <0–100>,
  "overall_assessment": "<summary>",
  "key_strengths": "<strengths>",
  "improvement_areas": "<areas for improvement>"
}}
"""

In [35]:
# ================================================
# Phase 5 — Append-Only Judging (Preserve ALL Phase-4 Metadata)
# ================================================
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from copy import deepcopy
import json
import os

# -----------------------------
# Helpers
# -----------------------------
def newest(path_iter):
    items = sorted([p for p in path_iter], key=lambda p: p.stat().st_mtime, reverse=True)
    return items[0] if items else None

def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def safe_json(s: str):
    try:
        return json.loads(s)
    except Exception:
        return None

def extract_student_response(sample_row):
    """
    Robustly extract resps[0], handling [["text"]] and ["text"].
    """
    resps = sample_row.get("resps")
    if not resps:
        return ""
    first = resps[0]
    if isinstance(first, list) and first:
        first = first[0]
    return first if isinstance(first, str) else ""

def derive_prompt_fields(phase4_row):
    """
    Build prompt fields from the full Phase-4 row.
    Does not mutate the row; only reads.
    """
    doc = (phase4_row.get("doc") or {}) if isinstance(phase4_row.get("doc"), dict) else {}
    osq_question    = doc.get("osq_prompt") or doc.get("question") or ""
    expected_answer = doc.get("expected_answer", "")
    blooms_level    = doc.get("blooms_level", "N/A")
    se_domain       = doc.get("INCOSE Handbook Category", "General Systems Engineering")
    student_resp    = extract_student_response(phase4_row)
    return {
        "osq_question": osq_question,
        "expected_answer": expected_answer,
        "student_response": student_resp,
        "blooms_level": blooms_level,
        "se_domain": se_domain,
    }

def triad_missing(fields):
    """
    Ensure the key triad exists for judging: question, expected, student_response.
    """
    missing = [k for k in ("osq_question", "expected_answer", "student_response")
               if not isinstance(fields.get(k, ""), str) or not fields.get(k, "").strip()]
    return missing

def ensure_alignment_or_die(existing_row, current_src_row, sample_id):
    """
    If final Phase-5 file already contains this sample_id, ensure its
    'phase4_row' snapshot matches Phase-4 source RIGHT NOW.
    This compares the ENTIRE phase4_row (deep equality).
    """
    snap = existing_row.get("phase4_row")
    if snap is None:
        raise RuntimeError(
            f"[ALIGNMENT ERROR] sample_id={sample_id} has no 'phase4_row' snapshot in Phase-5 file."
        )
    if snap != current_src_row:
        raise RuntimeError(
            f"\n[ALIGNMENT ERROR] sample_id={sample_id} Phase-4 content changed.\n"
            f"Refusing to proceed. Freeze Phase-4 or regenerate Phase-5 from scratch."
        )

# -----------------------------
# Preconditions
# -----------------------------
if not PHASE4_ROOT.exists():
    raise FileNotFoundError(f"Phase-4 task directory not found: {PHASE4_ROOT}")

model_dirs = [d for d in PHASE4_ROOT.iterdir() if d.is_dir()]
if not model_dirs:
    raise FileNotFoundError(f"No model directories under {PHASE4_ROOT}")

print(f"Discovered {len(model_dirs)} model directories under task '{TASK_NAME}'.")

# -----------------------------
# Main loop per model
# -----------------------------
for model_dir in model_dirs:
    model_name = model_dir.name
    src_samples = newest(model_dir.glob("samples_*.jsonl"))
    if not src_samples:
        print(f"[skip] {model_name}: missing samples_*.jsonl")
        continue

    # Load Phase-4 samples (source of truth)
    source_rows = load_jsonl(src_samples)
    if SAMPLE_N > 0:
        source_rows = source_rows[:SAMPLE_N]
    total = len(source_rows)
    print(f"\nModel: {model_name} | Source: {src_samples.name} | Count: {total}")

    # Phase-5 output dir mirrors task/model
    out_dir = PHASE5_ROOT / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # Decide final Phase-5 samples file (verbatim Phase-4 name + judge suffix)
    phase4_name = src_samples.name.replace(".jsonl", "")
    judge_suffix = JUDGE_MODEL.replace("/", "_")
    target_name = f"{phase4_name}__{judge_suffix}.jsonl"
    final_samples_path = out_dir / target_name

    # If it already exists, resume. If not, create empty file.
    if final_samples_path.exists():
        print(f"[resume] Reusing existing samples file: {final_samples_path.name}")
    else:
        with open(final_samples_path, "w", encoding="utf-8") as _:
            pass
        print(f"[new] Creating samples file: {final_samples_path.name}")

    # Resume: read existing judged rows
    done_ids = set()
    existing_by_id = {}
    existing_rows = load_jsonl(final_samples_path) if final_samples_path.exists() else []
    for row in existing_rows:
        sid = row.get("sample_id")
        if isinstance(sid, int):
            done_ids.add(sid)
            existing_by_id[sid] = row

    # Alignment check for already-judged IDs (deep compare of the entire Phase-4 row)
    for sid in sorted(done_ids):
        if sid >= total:
            raise RuntimeError(f"{final_samples_path.name} has sample_id {sid} beyond current source length {total}.")
        ensure_alignment_or_die(existing_by_id[sid], source_rows[sid], sid)

    print(f"[progress] {model_name}: {len(done_ids)} already judged, {total - len(done_ids)} remaining.")

    # Judge remaining samples (append line-by-line)
    to_do = [i for i in range(total) if i not in done_ids]
    if not to_do:
        print(f"[done] {model_name}: nothing to judge.")
        continue

    with open(final_samples_path, "a", encoding="utf-8") as fout:
        for sid in tqdm(to_do, desc=f"Judging {model_name}", unit="sample"):
            phase4_row = deepcopy(source_rows[sid])  # preserve ALL meta from Phase-4
            fields_for_prompt = derive_prompt_fields(phase4_row)

            # If core triad is missing, write a SKIPPED record but keep full Phase-4 snapshot
            missing = triad_missing(fields_for_prompt)
            if missing:
                record = {
                    "sample_id": sid,
                    "phase4_row": phase4_row,  # full snapshot for later mapping (MCQ↔OSQ, etc.)
                    "judge": {
                        "fields": {
                            "technical_accuracy":      {"score": None, "justification": None},
                            "conceptual_understanding":{"score": None, "justification": None},
                            "completeness":            {"score": None, "justification": None},
                            "clarity_organization":    {"score": None, "justification": None},
                            "professional_relevance":  {"score": None, "justification": None},
                            "overall_score": None,
                            "overall_assessment": f"SKIPPED: missing fields {missing}",
                            "key_strengths": None,
                            "improvement_areas": None,
                        },
                        "prompt": None,
                        "raw_output": None,
                        "timestamp": datetime.now().isoformat(),
                        "meta": {
                            "judge_model": JUDGE_MODEL,
                            "temperature": TEMPERATURE,
                            "max_tokens": MAX_TOKENS,
                            "task_name": TASK_NAME,
                            "model_name": model_name
                        }
                    }
                }
                fout.write(json.dumps(record) + "\n"); fout.flush()
                continue

            # Build prompt
            prompt = JUDGE_PROMPT.format(**fields_for_prompt)

            # Call judge LLM
            raw = None
            parsed = None
            api_error = None
            try:
                completion = client.chat.completions.create(
                    model=JUDGE_MODEL,
                    temperature=TEMPERATURE,
                    max_tokens=MAX_TOKENS,
                    messages=[
                        {"role": "system", "content": "You are an expert evaluator in systems engineering education."},
                        {"role": "user", "content": prompt}
                    ]
                )
                raw = (completion.choices[0].message.content or "").strip()
                parsed = safe_json(raw)
            except Exception as e:
                api_error = str(e)

            # Build judge fields (strict but resilient)
            fields = {
                "technical_accuracy":      {"score": None, "justification": None},
                "conceptual_understanding":{"score": None, "justification": None},
                "completeness":            {"score": None, "justification": None},
                "clarity_organization":    {"score": None, "justification": None},
                "professional_relevance":  {"score": None, "justification": None},
                "overall_score": None,
                "overall_assessment": None,
                "key_strengths": None,
                "improvement_areas": None,
            }
            if parsed is None:
                fields["overall_assessment"] = "ERROR: Invalid JSON response" + (f" ({api_error})" if api_error else "")
            else:
                def sget(d, key):
                    v = d.get(key)
                    return (v or {}).get("score") if isinstance(v, dict) else v
                fields["technical_accuracy"]       = {"score": sget(parsed,"technical_accuracy"),      "justification": (parsed.get("technical_accuracy") or {}).get("justification")}
                fields["conceptual_understanding"] = {"score": sget(parsed,"conceptual_understanding"),"justification": (parsed.get("conceptual_understanding") or {}).get("justification")}
                fields["completeness"]             = {"score": sget(parsed,"completeness"),            "justification": (parsed.get("completeness") or {}).get("justification")}
                fields["clarity_organization"]     = {"score": sget(parsed,"clarity_organization"),    "justification": (parsed.get("clarity_organization") or {}).get("justification")}
                fields["professional_relevance"]   = {"score": sget(parsed,"professional_relevance"),  "justification": (parsed.get("professional_relevance") or {}).get("justification")}
                fields["overall_score"]            = parsed.get("overall_score")
                fields["overall_assessment"]       = parsed.get("overall_assessment")
                fields["key_strengths"]            = parsed.get("key_strengths")
                fields["improvement_areas"]        = parsed.get("improvement_areas")

            # Append one JSON object per judged sample directly to the final samples file
            record = {
                "sample_id": sid,
                # FULL Phase-4 row preserved here for later mapping/joins (MCQ↔OSQ, taxonomy, etc.)
                "phase4_row": phase4_row,
                "judge": {
                    "fields": fields,
                    "prompt": prompt,
                    "raw_output": raw,
                    "timestamp": datetime.now().isoformat(),
                    "meta": {
                        "judge_model": JUDGE_MODEL,
                        "temperature": TEMPERATURE,
                        "max_tokens": MAX_TOKENS,
                        "task_name": TASK_NAME,
                        "model_name": model_name
                    }
                }
            }
            fout.write(json.dumps(record) + "\n")
            fout.flush()

    print(f"[✓] Samples appended for {model_name}: {final_samples_path.name}")

print("\nAll models processed.")


Discovered 2 model directories under task 'sysengbench-osq'.

Model: gemma3__27b | Source: samples_sysengbench-osq_2025-10-01T04-26-37.597737.jsonl | Count: 3
[new] Creating samples file: samples_sysengbench-osq_2025-10-01T04-26-37.597737__openai_gpt-5.jsonl
[progress] gemma3__27b: 0 already judged, 3 remaining.


Judging gemma3__27b: 100%|██████████| 3/3 [00:51<00:00, 17.26s/sample]

[✓] Samples appended for gemma3__27b: samples_sysengbench-osq_2025-10-01T04-26-37.597737__openai_gpt-5.jsonl

Model: gemma3__4b | Source: samples_sysengbench-osq_2025-10-01T03-58-29.887651.jsonl | Count: 3
[resume] Reusing existing samples file: samples_sysengbench-osq_2025-10-01T03-58-29.887651__openai_gpt-5.jsonl
[progress] gemma3__4b: 3 already judged, 0 remaining.
[done] gemma3__4b: nothing to judge.

All models processed.


# Note that no `results_` file is made. This is INTENTIONAL. Process everything in Phase 6 for analysis.
The results file would only include final accuracy. We can process that prior to analysis. 